# Controlled ResNet50 fold-0 pilot — PREPARATION ONLY
Do not upload or train until the owner separately approves cloud processing and this run.
544 fit / 136 score, all original Training. No Validation, Test, held records or personal photos.
Provisional grades 1–4; no whole-image grade-0 support. Content groups are isolated;
participant independence and clinical labels are unverified. This is not gingiva-supervised training.

After approval: choose a CUDA GPU (T4 is sufficient), set permission flags below, install
dependencies, manually upload the named ZIP using the Files sidebar, then verify/extract.
No Drive mount or automatic image uploader is used. Keep the notebook private; omit outputs
when saving/sharing because later CAM views contain dental images.


In [ ]:
# Keep both False during preparation. Run All cannot start training by default.
import os, sys
from pathlib import Path
sys.dont_write_bytecode = True
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
CLOUD_UPLOAD_APPROVED = False
RUN_TRAINING = False
OWNER_APPROVAL = ""
ARCHIVE = Path("/content/resnet50_fold0_pilot_v1.zip")
ROOT = Path("/content/resnet50_fold0_package_v1")
RUN_ROOT = Path("/content/resnet50_fold0_run_v1")
ARCHIVE_SHA = "60f70f2703a95c870586190705e991ca72dfa6a2e999096a554585b2433665fa"
MANIFEST_SHA = "1d2a49284a885621aa3b89d059f2be6283f381aa9449db0c4ead8eca206ebec4"


In [ ]:
# Install only after separate approval, before importing Torch/NumPy.
if not CLOUD_UPLOAD_APPROVED:
    raise PermissionError("STOP: no external upload or GPU run is authorized yet.")
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch==2.11.0", "torchvision==0.26.0", "numpy==2.5.3",
    "Pillow==11.3.0", "scikit-learn==1.9.1"])
# Restart the session if Colab reports that imported packages must be restarted.


## Verify the manually uploaded ZIP before importing its code
Expected filename: `resnet50_fold0_pilot_v1.zip`. The ZIP includes the pinned public
ImageNet initialization and local dataset derivatives; no old dental model checkpoint.
Extraction refuses an existing destination. To resume a training session, reuse the already
verified package and matching run directory; do not re-extract into it.


In [ ]:
if not CLOUD_UPLOAD_APPROVED:
    raise PermissionError("Cloud processing has not been approved")
import hashlib, json, stat, zipfile
def file_sha(path):
    result = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024*1024), b""):
            result.update(block)
    return result.hexdigest()
assert file_sha(ARCHIVE) == ARCHIVE_SHA, "Wrong or damaged ZIP"
if not ROOT.exists():
    with zipfile.ZipFile(ARCHIVE) as archive:
        entries = archive.infolist()
        names = [entry.filename for entry in entries]
        assert len(names) == len(set(names)) < 750
        assert sum(entry.file_size for entry in entries) < 1024**3
        for entry in entries:
            name = Path(entry.filename)
            assert not name.is_absolute() and ".." not in name.parts
            assert not entry.is_dir() and not stat.S_ISLNK(entry.external_attr >> 16)
            assert entry.file_size < 110*1024**2
        assert hashlib.sha256(archive.read("manifest.json")).hexdigest() == MANIFEST_SHA
        manifest = json.loads(archive.read("manifest.json"))
        assert set(names) == set(manifest["files"]) | {"manifest.json"}
        ROOT.mkdir()
        for name in names:
            member = ROOT / name
            member.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(name) as source, member.open("xb") as destination:
                import shutil
                shutil.copyfileobj(source, destination)
assert file_sha(ROOT / "manifest.json") == MANIFEST_SHA
# Check vendored source hashes before the first import, even on a resumed session.
manifest = json.loads((ROOT / "manifest.json").read_text())
for name, info in manifest["files"].items():
    path = ROOT / name
    assert not path.is_symlink() and file_sha(path) == info["sha256"], name
sys.path.insert(0, str(ROOT))
from src.vision import resnet50_fold0_pilot_v1 as pilot
PREFLIGHT = pilot.preflight(ROOT, MANIFEST_SHA)
print(json.dumps(PREFLIGHT, indent=2))


In [ ]:
# GPU/runtime check: no fitting occurs in this cell.
import torch, torchvision, numpy, sklearn, PIL
print({"torch": torch.__version__, "torchvision": torchvision.__version__,
       "numpy": numpy.__version__, "sklearn": sklearn.__version__, "Pillow": PIL.__version__})
assert torch.cuda.is_available(), "CUDA GPU required; stop instead of CPU training"
print(torch.cuda.get_device_name(), "CUDA", torch.version.cuda)


## Explicit future execution gate
After a separate owner approval, set RUN_TRAINING=True and OWNER_APPROVAL to:
`I approve this exact fold-0 Colab GPU pilot and cloud processing.`

Budget: shared warm-up 1 epoch, arm A 5 epochs, arm B 5 epochs; no early stopping,
score-driven selection or automatic extra epochs. Fresh Adam per stage; same arm batch orders.
Checkpoints are saved every epoch. Identical-runtime interrupted fits can resume with the
same RUN_ROOT. An incomplete checkpoint is preserved and fails closed. A completed run refuses
execution. Scoring/CAM export interruptions require inspection; they are not silently overwritten.


In [ ]:
if not CLOUD_UPLOAD_APPROVED or not RUN_TRAINING:
    raise PermissionError("Preparation complete. STOP and obtain explicit training approval.")
RESULT = pilot.run(ROOT, RUN_ROOT, execute=RUN_TRAINING,
                   approval=OWNER_APPROVAL, manifest_sha=MANIFEST_SHA)
print(json.dumps(RESULT, indent=2))


## Review and return results — only after a completed approved run
Review `metrics.json`, `predictions.json`, `warmup/`, `A/`, `B/` and `cam/`.
CAM contains predicted-class and common-A-target original/heatmap/overlay views for the same
eight predeclared photos. Attention is not validated disease localization or gum boundaries.
Doctor review should be blinded to model identity; record relevant/irrelevant/uncertain and
notes separately. No attention-mask score is claimed without qualified clinical annotation.

The optional return ZIP contains experimental CAM images as well as weights/results.
It stays on the Colab VM until you explicitly download it. It never installs into the PP1 demo.
Colab local storage is ephemeral: manually download checkpoints/results before ending the session.


In [ ]:
# Explicit optional packaging; never upload or install into the Mac demo automatically.
assert (RUN_ROOT / "complete.json").is_file(), "No completed experiment yet"
import shutil
RETURN_BASE = Path("/content/resnet50_fold0_return_v1")
assert not RETURN_BASE.with_suffix(".zip").exists(), "Preserve existing return ZIP"
files = {path.relative_to(RUN_ROOT).as_posix(): file_sha(path)
         for path in RUN_ROOT.rglob("*") if path.is_file()}
integrity = RUN_ROOT / "return_integrity.json"
assert not integrity.exists(), "Preserve existing integrity receipt"
integrity.write_text(json.dumps(files, indent=2))
shutil.make_archive(str(RETURN_BASE), "zip", RUN_ROOT)
print("Download manually:", RETURN_BASE.with_suffix(".zip"))
print("SHA-256:", file_sha(RETURN_BASE.with_suffix(".zip")))
